### **Appendix: Safe Reinforcement Learning with a Safety Filter**

In Chapters 7 and 8, we learned control policies purely from interaction with the system. Learning requires exploration, and an exploratory action can potentially drive the system into a state that must be avoided, such as a collision with a wall. **Safe reinforcement learning** studies how a system can be kept safe while it is still learning. In this appendix, we use a **safety filter** for this purpose. The filter is a model-based layer between the learner and the system that checks every action before it is applied and replaces it whenever it would eventually lead the system out of a safe set.

Instead of the mountain car, we consider the **cart-pole swing-up** example, a classic benchmark in which a pole mounted on a cart has to be swung up from its hanging position and then balanced upright. We proceed in three steps. First, we train a **deep Q-network (DQN)** on the swing-up task without any safety mechanism. Second, we construct a **safety filter** that keeps the cart within prescribed bounds. It is based on the **maximal control-invariant set** of the constraint, which we compute offline with Hamilton-Jacobi reachability. Finally, we introduce the double-sided constraints to this example, train the same DQN with the filter in the loop, and compare both learners over 10 random seeds.

**Note that** this example is the safe reinforcement learning example of the tutorial paper *"A Decade of Learning-Based Control:
From Theoretic Foundations to Practical Deployment"*, and the last section of this notebook reproduces its two figures. The DQN implementation are adpated from the course material [*Reinforcement Learning and Learning-based Control*](https://github.com/Data-Science-in-Mechanical-Engineering/RLLBC) (DSME, RWTH Aachen University).

First, we need to set up our Python environment and import relevant packages.

In [ ]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from appendix.safety_filter import CartPoleSwingUp, make_filter, check_invariance, DEFAULT as SAFETY
from appendix.dqn_utils import *

<br>

----

<br>

### **The Cart-Pole Swing-Up Setup**

The system consists of a cart that moves along a track of limited length, $|x| \leq 2.4\,m$, and a pole that is hinged on the cart. Its state is $\boldsymbol{s} = [x, \dot{x}, \theta, \dot{\theta}]^\top$, where $\theta = 0$ corresponds to the upright pole. The dynamics are those of the classic cart-pole system, i.e., the same equations as in gymnasium's `CartPole-v1`, discretized with the explicit Euler method and a time step of $\tau = 0.02\,s$. The swing-up task differs from the classic balancing task in the following aspects:

- **Initial state.** The pole starts in the hanging position ($\theta \approx \pi$) and has to be swung up before it can be balanced.
- **Actions.** In every step, the agent applies one of three forces $F \in \{-10\,N, 0, 10\,N\}$ to the cart.
- **Reward.** The reward is dense. It is the product of three factors that measure how upright the pole is, how close the cart is to the center of the track, and how slowly the pole rotates. A balanced pole in the middle of the track thus yields the maximal reward of 1 per step.
- **End of an episode.** A falling pole does not end the episode. An episode only terminates when the cart leaves the track, and it is truncated after 500 steps (10 s). The return is therefore at most 500.

The reward is given by

$$r = \frac{1 + \cos\theta}{2} \cdot \left(1 - \left(\frac{x}{2.4}\right)^2\right) \cdot \frac{1 + 10^{-(\dot{\theta}/5)^2}}{2} \in [0, 1].$$

The network receives the observation $(x, \dot{x}, \cos\theta, \sin\theta, \dot{\theta})$, which avoids the discontinuity of the angle at $\pm\pi$. The class `CartPoleSwingUp` implements the environment with the standard `reset`/`step` interface:

In [ ]:
env = CartPoleSwingUp()
obs, _ = env.reset(seed=0)
print(f"forces: {env.forces} N")
print(f"initial observation (x, x_dot, cos(theta), sin(theta), theta_dot): {np.round(obs, 3)}")
obs, reward, terminated, truncated, _ = env.step(2)
print(f"after one push to the right: reward {reward:.4f}, terminated {terminated}, truncated {truncated}")

<br>

----

<br>

### **Example 1.1: Deep Q-Network (DQN)**

Recall Q-learning from Chapter 7.2, which learns the optimal action-value function from individual transitions $(s, a, r, s')$ with the update

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left( r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right).$$

Since the state of the cart-pole is continuous, a tabular representation of $Q$ is no longer possible. **DQN** (Mnih et al., 2015) therefore approximates $Q$ by a neural network $Q_\phi(s, a)$ with one output per action and stabilizes the training with two additional components:

- **Replay buffer.** All transitions are stored in a buffer, and every gradient step uses a mini-batch sampled uniformly from it. This breaks the correlation between consecutive samples and allows each transition to be reused many times, which is possible because DQN is an *off-policy* method.
- **Target network.** A copy $Q_{\phi^-}$ of the network, which is only updated every 500 steps, provides the regression target.

The target and the loss over a mini-batch $B$ read

$$y = r + \gamma\, (1 - d)\, \max_{a'} Q_{\phi^-}(s', a'), \qquad L(\phi) = \frac{1}{|B|} \sum_{(s, a, r, s', d) \in B} \big(y - Q_\phi(s, a)\big)^2,$$

where $d = 1$ only if the episode has *terminated*, i.e., if the cart has left the track. The time limit of 500 steps is not part of the task, so the target is still bootstrapped after a truncated episode.

The agent explores with an **$\epsilon$-greedy** policy. With probability $\epsilon$, it applies a random force, and otherwise the force with the largest Q-value. The exploration rate $\epsilon$ decreases linearly from 1 to 0.05 over the first 30 % of the interactions. During training, the greedy policy is evaluated every 1000 interactions and every 25 episodes, each time on 10 episodes, and the mean return of these episodes yields the learning curves below.

The implementation is provided in `dqn_utils.py` (`q_network`, `ReplayBuffer`, `train`) and uses the following hyperparameters:

In [ ]:
for name, value in HYPERPARAMS.items():
    print(f"{name:26s} {value}")

We now train a single agent, which takes one to two minutes on one CPU core:

In [ ]:
run_dqn = train(seed=1)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(run_dqn["steps"], run_dqn["returns"].mean(axis=1), color="tab:blue", label="DQN, seed 1")
ax.set(xlabel="environment interactions", ylabel="evaluation return", ylim=(-10, 510))
ax.grid(alpha=0.3)
ax.legend()
plt.show()

Before the first gradient step, the agent collects 10,000 interactions with random actions. It then learns to swing the pole up and, later, to balance it in the middle of the track. The learning curve is noisy, since each evaluation comprises only 10 episodes and a policy that balances the pole well may still drive the cart off the track from some initial states.

<br>

----

<br>

### **Example 2.1: A Safety Filter for the Cart Position**

We now require the cart to stay within the interval

$$x_{\min} \leq x \leq x_{\max}, \qquad x_{\min} = -1\,\text{m}, \quad x_{\max} = 2.4\,\text{m}.$$

The upper bound coincides with the end of the track, whereas the lower bound represents an additional wall. The pole itself is not constrained. In the safe variant of the task, leaving the interval terminates the episode and incurs a penalty.

**Why a one-step check is not sufficient.** A cart that approaches the wall at high speed cannot be stopped within a single step, since the force is limited to $10\,N$. The filter therefore needs the set of states from which *some* admissible input keeps the cart inside the interval for all future times. This set is called the **maximal control-invariant set**. With the constraint function $l(\boldsymbol{s}) = \min(x - x_{\min},\, x_{\max} - x)$, the value function

$$V(\boldsymbol{s}) = \max_{F(\cdot)} \; \min_{t \geq 0} \; l\big(\boldsymbol{s}(t)\big)$$

is the solution of a Hamilton-Jacobi-Isaacs partial differential equation, and its zero super-level set $\mathcal{S} = \{\boldsymbol{s} : V(\boldsymbol{s}) \geq 0\}$ is exactly the maximal control-invariant set. We compute $V$ **offline** on a $41 \times 41 \times 48 \times 41$ grid with the toolbox `hj_reachability`, which takes about 12 minutes on a CPU, and provide the result with this notebook in `safe_sets/`.

**Online filtering.** In every step, the filter predicts the successor state $\boldsymbol{s}'_F$ for each of the three forces with the model. A force is regarded as safe if $V(\boldsymbol{s}'_F) \geq \varepsilon$, where the margin $\varepsilon = 0.05$ accounts for the interpolation error of the grid and for the time discretization. If the intended force is safe, it is applied unchanged. Otherwise, the filter applies the force with the largest value $V(\boldsymbol{s}'_F)$. For states in which the pole rotates faster than the grid covers, the filter falls back to a high-order control barrier function (CBF) on the same bounds.

**Reward shaping.** If the learner only observed the filtered actions, it could not learn which of its own decisions were unsafe. Following CBF-RL ([Yang et al., 2025](https://arxiv.org/abs/2510.14959)), the reward is therefore augmented by two terms that vanish whenever the intended force is safe:

$$\tilde{r} = r + w_v \, \mathrm{clip}\!\left(\frac{\min\big(V(\boldsymbol{s}'_{F_\text{intent}}) - \varepsilon,\, 0\big)}{V_{\max}}, -1, 0\right) + w_d \left( \exp\!\left(-\frac{(F_\text{intent} - F_\text{safe})^2}{\sigma^2}\right) - 1 \right),$$

with $w_v = w_d = 2$ and $\sigma = 10\,N$. In addition, the **replay buffer stores the intended action** while the environment applies the safe one, which keeps the off-policy update consistent with the filtered system. Both components are implemented in `safety_filter.py` (`SafetyFilter`, `SafetyWrapper`).

In the following, we load the invariant set and visualize a slice of it for the pole hanging down at rest:

In [ ]:
safety_filter = make_filter(verbose=True)
safe_set = safety_filter.set
print(f"safe fraction of the grid: {safe_set.safe_fraction():.2f}, "
      f"V at the initial state: {safe_set.interpolate(np.array([[0.0, 0.0, np.pi, 0.0]]))[0]:.2f}")

# slice through the set for the pole hanging down at rest
x, x_dot = np.meshgrid(np.linspace(-1.3, 2.7, 400), np.linspace(-5, 5, 400))
states = np.column_stack([x.ravel(), x_dot.ravel(), np.full(x.size, np.pi), np.zeros(x.size)])
value = safe_set.interpolate(states).reshape(x.shape)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.contourf(x, x_dot, value >= 0, levels=[-0.5, 0.5, 1.5], colors=["#f4cccc", "#d9ead3"])
ax.contour(x, x_dot, value, levels=[0], colors="k", linewidths=1)
for bound in (SAFETY["x_min"], SAFETY["x_max"]):
    ax.axvline(bound, color="tab:red", ls="--", lw=1)
ax.set(title=r"pole hanging down, $\dot\theta = 0$: green = invariant set",
       xlabel=r"cart position $x$ (m)", ylabel=r"cart velocity $\dot x$ (m/s)")
plt.show()

The boundary of the set is a **braking curve**. The closer the cart is to a bound, the lower its velocity towards that bound must be, because the force available for braking is limited. Note that the invariant set is strictly smaller than the interval itself. States that lie within the bounds but beyond the braking curve cannot be kept inside the interval by any admissible input.

Before the filter is used for learning, we verify its invariance property. Starting from random states inside the set, we simulate 100,000 transitions each with random intended actions and with adversarial intended actions that always push the cart in the same direction:

In [ ]:
for mode in ("random", "right", "left"):
    print(f"{mode:>6s} intents:", check_invariance(mode=mode))

In none of the transitions does the state leave the set or does the cart cross a bound, and at least one safe force is always available (`infeasible` = 0). A small number of transitions leave the grid in $\dot{\theta}$ and are handled by the CBF fallback (`off_grid`).

<br>

----

<br>

### **Example 2.2: Safe DQN**

Safe DQN uses exactly the same learner and hyperparameters as DQN, but interacts with the filtered environment `SafetyWrapper(CartPoleSwingUp())`, which is selected with `train(..., safe=True)`. Its policy is evaluated in the same way as the policy of DQN, i.e., on the plain task **without bounds and without the filter**, so that the learning curves of both learners are directly comparable.

Since the outcome of a single run depends strongly on the random seed, we compare the two learners over 10 seeds each, as in the tutorial paper, with 350,000 interactions per run. Running all 20 trainings in parallel on 10 CPU cores takes about 5 minutes. Their evaluation results are provided in `checkpoints/`, and setting `RETRAIN = True` repeats the training. 

For each learner, we report the number of interactions until the mean evaluation return first reaches 350, the area under the learning curve normalized by the number of interactions, and the best and the final return.

In [ ]:
ARM_NAMES = {"dqn": "DQN", "safe_dqn": "Safe DQN"}
SHOWN_SEED = 3                                                        # the policies shown in the paper figures
SHOWN_POLICIES = [("dqn", 210000), ("dqn", 294000), ("safe_dqn", 120000)]

RETRAIN = False
if RETRAIN:
    new_runs = train_all(seeds=range(1, 11), snapshot_steps=[step for _, step in SHOWN_POLICIES])
    save_runs("checkpoints/dqn_learning_curves.npz", new_runs)
    for arm, step in SHOWN_POLICIES:
        torch.save(new_runs[(arm, SHOWN_SEED)]["policies"][step], f"checkpoints/{arm}_seed{SHOWN_SEED}_{step // 1000}k.pt")

runs = load_runs("checkpoints/dqn_learning_curves.npz")
policies = {(arm, step): load_policy(f"checkpoints/{arm}_seed{SHOWN_SEED}_{step // 1000}k.pt")
            for arm, step in SHOWN_POLICIES}

print(f"{'':9s} {'reached 350':>12s} {'interactions to 350':>21s} {'area under curve':>18s} {'best return':>13s} {'final return':>14s}")
for arm, r in summary(runs).items():
    (s, s_sd), (a, a_sd), (b, b_sd), (f, f_sd) = r["steps_to_threshold"], r["auc"], r["best"], r["final"]
    print(f"{ARM_NAMES[arm]:9s} {r['reached']:>9d}/{r['seeds']:<2d} {s:>12,.0f} ± {s_sd:<7,.0f}"
          f"{a:>10.0f} ± {a_sd:<5.0f} {b:>7.0f} ± {b_sd:<3.0f} {f:>8.0f} ± {f_sd:<3.0f}")
print(f"\nCart outside [{SAFETY['x_min']}, {SAFETY['x_max']}] during training (10 seeds x 350,000 interactions):")
for arm, name in ARM_NAMES.items():
    per_seed = [r for (a, _), r in runs.items() if a == arm]
    steps_outside = sum(int(r['steps_outside']) for r in per_seed)
    print(f"  {name:9s} {sum(int(r['violations']) for r in per_seed):>6,d} of {sum(int(r['episodes']) for r in per_seed):,d} episodes "
          f"({steps_outside:,d} interaction{'s' if steps_outside != 1 else ''})")

fig, ax = plt.subplots(figsize=(8, 3.5))
for arm, color in (("dqn", "tab:blue"), ("safe_dqn", "tab:green")):
    grid, mean, std, n = learning_curves(runs, arm)
    ax.fill_between(grid, mean - std, mean + std, color=color, alpha=0.2, linewidth=0)
    ax.plot(grid, mean, color=color, label=f"{ARM_NAMES[arm]} (mean ± std over {n} seeds)")
ax.axhline(350, color="0.4", ls=":", lw=1)
ax.set(xlabel="environment interactions", ylabel="evaluation return")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
plt.show()

#### **Result Analysis:**

- **Safety during learning.** In 3.5 million training interactions, including the initial phase in which the agent acts completely at random, the cart of Safe DQN leaves the interval only once (seed 10). Without the filter, the cart of DQN leaves the interval in 4,700 of 8,337 training episodes, almost always beyond the lower bound. DQN is not aware of the constraint, since the wall at $x_{\min}$ is not part of its task, so these numbers show how often unconstrained exploration enters the region that the filter protects. The safety provided by the filter does not depend on the quality of the policy but only on the model and the invariant set. The single violation, however, illustrates the limits of this guarantee. At that moment, the pole was rotating at almost 15 rad/s, which is the edge of the grid. The only force that could still have braked the cart would have accelerated the pole beyond the grid, where $V$ is unknown. The filter therefore regarded this force as unsafe, and the state left the invariant set. A grid that covers faster pole rotations, or a CBF fallback for individual forces whose successor state leaves the grid, would close this gap.
- **Faster learning.** Safe DQN reaches a return of 350 after about 57,000 interactions instead of about 136,000, which is 2.4 times faster, and its area under the learning curve is 27 % larger. This holds even though its policy is evaluated on the plain task without the filter. Because the filter keeps the cart away from both ends of the track, every training episode of the safe learner lasts the full 500 steps, whereas many episodes of the plain DQN end early at the edge of the track. The safe learner can thus spend its interactions on the relevant part of the task, namely balancing the pole, and the reward shaping informs it about the intended actions that the filter had to correct.
- **Independence of the learned policy from the filter.** All evaluations are carried out on the plain task without the filter, so the higher returns are achieved by the policy alone. The filter supports the learning process, but the final result does not rely on it.

The price of this approach is the need for a model. The filter uses the dynamics both to compute the invariant set and to predict the successor states. As for the model-based controllers of Chapters 5 and 6, its guarantee therefore holds only as far as the model is accurate. In this example, the filter uses the exact dynamics.

<br>

----

### **Appendix: Figures for the Tutorial Paper**

The cell below redraws the results of this notebook at single-column width for the tutorial paper and saves them to `figures/`. It contains no new training. In each figure, the left panel shows the path of the pole tip during one greedy episode of the plain task (seed 3), and the right panel shows the evaluation return as mean ± standard deviation over the 10 seeds. The first figure shows DQN with two of its policies, and the second figure shows Safe DQN in color on top of the DQN reference in gray. Both figures share the same axis ranges, so that they can be compared directly.

In [ ]:
import matplotlib.ticker as mticker
import matplotlib.transforms as mtransforms

PAPER_RC = {"font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7,
            "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
            "axes.linewidth": 0.6, "lines.linewidth": 1.0,
            "font.family": "serif", "mathtext.fontset": "dejavuserif", "pdf.fonttype": 42, "ps.fonttype": 42,
            "legend.handlelength": 1.9, "legend.borderpad": 0.2, "legend.labelspacing": 0.25,
            "legend.columnspacing": 1.0, "legend.handletextpad": 0.5, "axes.grid": False}
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

CURVE_COLORS = {"dqn": "#0072B2", "safe_dqn": "#0B6E4F"}    # blue and dark green, colour-blind safe
POLICY_COLORS = ["#CC79A7", "#5D3A9B", "#4CC38A"]            # early DQN (pink), late DQN (purple), Safe DQN (light green)
REFERENCE_GREY, FORBIDDEN, SKETCH = "#9aa3ad", "#d62728", "#ccd2da"
ANN_FS, ZOOM = 5.8, 1.2                                       # annotation size as in 6.4, room around the paths
WIDTH, PANEL_HEIGHT = 3.5, 1.65                               # one column

# the greedy episode of each shown policy, all from the same initial state
labels, colors, paths, episode_returns, policy_steps = [], {}, {}, {}, {}
for i, (arm, step) in enumerate(SHOWN_POLICIES):
    label = f"{ARM_NAMES[arm]}, {step / 1000:.0f}k steps"
    tip_x, tip_y, episode_returns[label], pole_length = greedy_rollout(policies[(arm, step)], SHOWN_SEED)
    labels.append(label)
    colors[label], paths[label], policy_steps[label] = POLICY_COLORS[i], (tip_x, tip_y), step
    print(f"{label:22s} episode return {episode_returns[label]:6.1f}")
best_dqn = max(labels[:2], key=episode_returns.get)          # the DQN policy drawn in grey behind the safe one
safe = labels[2]
extent = (max(np.abs(x).max() for x, _ in paths.values()), max(np.abs(y).max() for _, y in paths.values()))
curves = {arm: learning_curves(runs, arm) for arm in ARM_NAMES}


def draw_curve(ax, arm, color, zorder=3, in_legend=True):
    grid, mean, std, n = curves[arm]
    ax.fill_between(grid, mean - std, mean + std, color=color, alpha=0.2, linewidth=0, zorder=zorder - 1)
    ax.plot(grid, mean, color=color, zorder=zorder, label=f"{ARM_NAMES[arm]} ({n} seeds)" if in_legend else None)


def decorate_path(ax, forbidden=True):
    # a sketch of the cart, the start and target positions of the pole tip and, for the safe learner, the forbidden region
    tilt = np.deg2rad(35.0)
    ax.plot([0, pole_length * np.sin(tilt)], [0, pole_length * np.cos(tilt)], color=SKETCH, linewidth=2.5,
            solid_capstyle="round", zorder=0)
    ax.add_patch(plt.Rectangle((-0.25, -0.10), 0.5, 0.20, facecolor="#eef1f5", edgecolor=SKETCH,
                               linewidth=1, joinstyle="round", zorder=0))
    for wheel_x in (-0.14, 0.14):
        ax.add_patch(plt.Circle((wheel_x, -0.145), 0.045, facecolor="#eef1f5", edgecolor=SKETCH, linewidth=1, zorder=0))
    ax.add_patch(plt.Circle((0, 0), 0.028, facecolor=SKETCH, edgecolor="none", zorder=0))
    ax.axhline(-0.19, color=SKETCH, linewidth=0.8, zorder=0)
    ax.plot([0, 0], [-pole_length, pole_length], linestyle="none", marker="o", color="black", markersize=2.6, zorder=6)
    annotations = [ax.annotate("target", (0, pole_length), textcoords="offset points", xytext=(0, 3),
                               ha="center", va="bottom", color="#4b5563", fontsize=ANN_FS),
                   ax.annotate("start", (0, -pole_length), textcoords="offset points", xytext=(0, -3),
                               ha="center", va="top", color="#4b5563", fontsize=ANN_FS)]
    blended = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)   # spans the panel, not autoscaled
    for bound, left in ((SAFETY["x_min"], True), (SAFETY["x_max"], False)) if forbidden else ():
        ax.add_artist(plt.Rectangle((bound - 1000 if left else bound, 0), 1000, 1, transform=blended,
                                    facecolor=FORBIDDEN, alpha=0.10, edgecolor="none", zorder=0))
        ax.axvline(bound, color=FORBIDDEN, linestyle="--", linewidth=0.9, alpha=0.9, zorder=1)
    ax.set_aspect("equal", adjustable="datalim")
    return annotations


def make_figure(path_entries, curve_entries, vlines, forbidden=True):
    # pole-tip paths on the left, learning curves on the right, one legend in a band underneath
    handles, texts = [], []
    if forbidden:
        handles.append(plt.Rectangle((0, 0), 1, 1, facecolor=FORBIDDEN, alpha=0.18, edgecolor=FORBIDDEN,
                                     linewidth=0.8, linestyle="--"))
        texts.append("cart not allowed")
    handles += [plt.Line2D([0], [0], color=color) for _, color, _ in path_entries]
    texts += [label for label, _, _ in path_entries]
    fig = plt.figure(figsize=(WIDTH, PANEL_HEIGHT + 0.13))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 0.13], width_ratios=[1, 1],
                          hspace=0.62, wspace=0.40, left=0.12, right=0.99, top=0.90, bottom=0.02)
    legend_ax = fig.add_subplot(gs[1, :])
    legend_ax.axis("off")
    ax_path, ax_ret = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
    for label, color, zorder in path_entries:
        ax_path.plot(*paths[label], color=color, zorder=zorder)
    for arm, color, zorder, in_legend in curve_entries:
        draw_curve(ax_ret, arm, color, zorder, in_legend)
    for step, color in vlines:
        ax_ret.axvline(step, color=color, linestyle="--", linewidth=0.9, alpha=0.85, zorder=1)
    annotations = decorate_path(ax_path, forbidden)
    ax_path.set(xlabel="horizontal position (m)", ylabel="height (m)")
    ax_ret.set(xlabel="environment interactions", ylabel="episode return")
    ax_path.set_title("Policy", pad=3)
    ax_ret.set_title("Learning curve", pad=3)
    ax_ret.set_xlim(min(g[0] for g, *_ in curves.values()), max(g[-1] for g, *_ in curves.values()))
    lo = min((m - sd).min() for _, m, sd, _ in curves.values())
    hi = max((m + sd).max() for _, m, sd, _ in curves.values())
    ax_ret.set_ylim(lo - 0.05 * (hi - lo), hi + 0.05 * (hi - lo))
    ax_ret.xaxis.set_major_locator(mticker.MultipleLocator(1e5))
    ax_ret.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: "0" if v == 0 else f"{v / 1e3:.0f}k"))
    ax_ret.yaxis.set_major_locator(mticker.MultipleLocator(200))     # the same ticks in both figures
    for h, t in zip(*ax_ret.get_legend_handles_labels()):
        handles.append(h)
        texts.append(t)
    band = legend_ax.get_position()
    fig.legend(handles, texts, loc="upper left", bbox_to_anchor=(0.0, band.y0, ax_ret.get_position().x1, band.height),
               mode="expand", ncol=len(texts), frameon=False, borderaxespad=0.0)
    return fig, ax_path, annotations


with plt.rc_context(PAPER_RC):
    built = [make_figure([(labels[0], colors[labels[0]], 3), (labels[1], colors[labels[1]], 3)],
                         [("dqn", CURVE_COLORS["dqn"], 3, True)],
                         [(policy_steps[label], colors[label]) for label in labels[:2]],
                         forbidden=False),
             make_figure([(safe, colors[safe], 4), (best_dqn, REFERENCE_GREY, 2)],
                         [("dqn", REFERENCE_GREY, 2, False), ("safe_dqn", CURVE_COLORS["safe_dqn"], 4, False)],
                         [(policy_steps[best_dqn], REFERENCE_GREY), (policy_steps[safe], colors[safe])])]

    # Both path panels get the same symmetric view around the cart. The two labels sit at a fixed offset in points,
    # so after the layout is final they are measured and the view is enlarged until they fit. With an equal aspect
    # and adjustable="datalim" the view follows the data limits, hence two invisible corner points set them.
    half_x, half_y = max(extent[0], pole_length) * ZOOM, max(extent[1], pole_length) * ZOOM
    for _ in range(4):
        for fig, ax_path, _ in built:
            ax_path.plot([-half_x, half_x], [-half_y, half_y], linestyle="none", marker="none", alpha=0)
            ax_path.set_aspect("equal", adjustable="datalim")
            fig.canvas.draw()
        reach = max(abs(v) for fig, _, annotations in built for a in annotations
                    for v in a.get_window_extent(fig.canvas.get_renderer()).transformed(a.axes.transData.inverted()).intervaly)
        if reach <= min(ax_path.get_ylim()[1] for _, ax_path, _ in built):
            break
        half_y = reach * 1.05
        half_x = max(half_x, half_y)

    for (fig, _, _), name in zip(built, ("tutorial_dqn", "tutorial_safe_dqn")):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")
        fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), bbox_inches="tight", dpi=300)
    plt.show()